In [1]:
import pandas as pd
import torch
import numpy as np
from numpy.linalg import norm

In [2]:
from transformers import (pipeline, AutoTokenizer, AutoModel)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
xlmr_sent = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

mbert_sent = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/bert-base-multilingual-cased-sentiment-multilingual",
    tokenizer="cardiffnlp/bert-base-multilingual-cased-sentiment-multilingual",
    truncation=True,
    device=0 if torch.cuda.is_available() else -1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


In [5]:
pairs = [
    {
        "id": 1,
        "archaic": "Məndə Mecnundan füzun aşiqlik istidadı var.",
        "modern": "Məndə Məcnundan da artıq aşiq olmaq qabiliyyəti var.",
        "gold_label": "positive"
    },
    {
        "id": 2,
        "archaic": "Məni candan usandırdı, cəfaindən usanmazmı?",
        "modern": "Mən əzabdan bezdim, bəs o zülm etməkdən bezməzmi?",
        "gold_label": "negative"
    },
    {
        "id": 3,
        "archaic": "Dərdimi deməyə bir həmdəm tapılmaz.",
        "modern": "Dərdimi bölüşəcəyim bir yaxın insan yoxdur.",
        "gold_label": "negative"
    },
    {
        "id": 4,
        "archaic": "Eşqdir ol kim, varlıqdan alar nişanı.",
        "modern": "Eşq insanın bütün varlığını dəyişdirən bir haldır.",
        "gold_label": "neutral"
    },
    {
        "id": 5,
        "archaic": "Aşiqlərə hər dəm bəladır səfa.",
        "modern": "Aşiqlər üçün rahatlıq belə əzab verir.",
        "gold_label": "negative"
    },
    {
        "id": 6,
        "archaic": "Könül kim, eşqə düşdü, səbrü qərar itirdi.",
        "modern": "Ürək eşqə düşəndə səbrini və rahatlığını itirir.",
        "gold_label": "negative"
    },
    {
        "id": 7,
        "archaic": "Bu hicran atəşi yandırdı məni.",
        "modern": "Bu ayrılıq məni daxildən yandırdı.",
        "gold_label": "negative"
    },
    {
        "id": 8,
        "archaic": "Sənsiz bu könül viranədir.",
        "modern": "Sənsiz ürəyim boş və dağılmış kimidir.",
        "gold_label": "negative"
    },
    {
        "id": 9,
        "archaic": "Eşq yolunda can vermək asandır.",
        "modern": "Sevgi uğrunda həyatını qurban vermək asandır.",
        "gold_label": "positive"
    },
    {
        "id": 10,
        "archaic": "Göz yaşı könlün sözüdür.",
        "modern": "Göz yaşları insanın hisslərini ifadə edir.",
        "gold_label": "neutral"
    },
    {
        "id": 11,
        "archaic": "Nola qan tökməkdə mahir olsa çeşmim mərdümü, Nütfeyi-qabildürür, qəmzən kimi ustadı var.",
        "modern": "Əgər mənim göz bəbəyim qan tökməkdə bu qədər ustadırsa, buna təəccüblənməmək lazımdır. Çünki o, öyrənməyə çox qabiliyyətli bir şagirddir və onun sənin baxışın kimi zalım və usta bir müəllimi var.",
        "gold_label": "negative"
    },
    {
        "id": 12,
        "archaic": "Bu cahan bir kölgədir, etibarı yox.",
        "modern": "Bu dünya keçicidir və ona güvənmək olmaz.",
        "gold_label": "negative"
    },
    {
        "id": 13,
        "archaic": "Qıl təfağür kim, sənin həm var mənim tək aşiqin, Leylinin Məcnunu, Şirinin əgər Fərhadı var.",
        "modern": "Elə bilmə tək sənə vurğun olan mənəm — Leylinin Məcnunu, Şirinin isə Fərhadı var.",
        "gold_label": "positive"
    },
    {
        "id": 14,
        "archaic": "Aşiqlik bir bəladır kim, ondan qaçmaq olmaz.",
        "modern": "Aşiqlik elə bir haldır ki, ondan qaçmaq mümkün deyil.",
        "gold_label": "negative"
    },
    {
        "id": 15,
        "archaic": "Könül sənsiz qərar tapmaz.",
        "modern": "Sənsiz ürəyim rahatlıq tapmır.",
        "gold_label": "negative"
    },
    {
        "id": 16,
        "archaic": "Bu dərd mənə həm yoldaş, həm də düşmən oldu.",
        "modern": "Bu dərd həm mənə yaxınlaşdı, həm də mənə zərər verdi.",
        "gold_label": "neutral"
    },
    {
        "id": 17,
        "archaic": "Hicran gecəsi gündüzdən uzun görünər.",
        "modern": "Ayrılıq gecəsi insana çox uzun gəlir.",
        "gold_label": "negative"
    },
    {
        "id": 18,
        "archaic": "Eşq sirrini faş edən özünə düşmən olar.",
        "modern": "Sevgi sirrini açan insan özünə zərər verir.",
        "gold_label": "negative"
    },
    {
        "id": 19,
        "archaic": "Dözməz könül bu qədər firqətə.",
        "modern": "Ürək bu qədər ayrılığa tab gətirmir.",
        "gold_label": "negative"
    },
    {
        "id": 20,
        "archaic": "Mən aşiqəm, məni rüsvay eyləmə.",
        "modern": "Mən səni sevirəm, məni biabır etmə.",
        "gold_label": "positive"
    },
    {
        "id": 21,
        "archaic": "Eşq oduna yananın ahı göylərə yetər.",
        "modern": "Sevgi əzabı çəkənin fəryadı çox böyük olur.",
        "gold_label": "negative"
    },
    {
        "id": 22,
        "archaic": "Bu eşq dərdi mənə min dərmandan əzizdir.",
        "modern": "Bu sevgi dərdi mənim üçün hər cür müalicədən dəyərlidir.",
        "gold_label": "positive"
    },
    {
        "id": 23,
        "archaic": "Aşiq üçün ölüm həyatın özü kimidir.",
        "modern": "Sevən insan üçün ölüm qorxulu deyil.",
        "gold_label": "positive"
    },
    {
        "id": 24,
        "archaic": "Eşq ağlı yoldan çıxarar.",
        "modern": "Sevgi insanın məntiqini əlindən alır.",
        "gold_label": "negative"
    },
    {
        "id": 25,
        "archaic": "Hər nəfəsdə bir ah çəkər aşiq.",
        "modern": "Aşiq insan hər an dərd içində olur.",
        "gold_label": "negative"
    },
    {
        "id": 26,
        "archaic": "Bu könül hicransız səfa bilməz.",
        "modern": "Bu ürək ayrılıq olmadan sevinci tanımır.",
        "gold_label": "negative"
    },
    {
        "id": 27,
        "archaic": "Eşq dərman istəməz, dərdin özüdür.",
        "modern": "Sevgi müalicə olunmur, özü bir dərddir.",
        "gold_label": "negative"
    },
    {
        "id": 28,
        "archaic": "Fəryadım aləmi tutdu, eşidən olmadı.",
        "modern": "Çox fəryad etdim, amma məni anlayan olmadı.",
        "gold_label": "negative"
    },
    {
        "id": 29,
        "archaic": "Bu bədən can ilədir, can isə eşq ilə.",
        "modern": "İnsan bədəni canla yaşayır, can isə sevgi ilə.",
        "gold_label": "positive"
    },
    {
        "id": 30,
        "archaic": "Könül verdim, əvəzində dərd aldım.",
        "modern": "Sevdim və qarşılığında əzab çəkdim.",
        "gold_label": "negative"
    },
    {
        "id": 31,
        "archaic": "Eşq yolunda səbr ən böyük silahdır.",
        "modern": "Sevgi yolunda səbr ən vacib gücdür.",
        "gold_label": "positive"
    },
    {
        "id": 32,
        "archaic": "Bu könül öz dərdindən özgə dərd tanımaz.",
        "modern": "Ürəyim yalnız öz ağrısını hiss edir.",
        "gold_label": "negative"
    },
    {
        "id": 33,
        "archaic": "Eşq məni məndən aldı.",
        "modern": "Sevgi məni əvvəlki halımdan çıxardı.",
        "gold_label": "negative"
    },
    {
        "id": 34,
        "archaic": "Hicran olmasa, vüsalın qədrin bilməzlər.",
        "modern": "Ayrılıq olmasa, qovuşma dəyərli olmaz.",
        "gold_label": "neutral"
    },
    {
        "id": 35,
        "archaic": "Aşiq üçün dünya dar olar.",
        "modern": "Sevən insan üçün dünya dar və sıxıcı görünür.",
        "gold_label": "negative"
    },
    {
        "id": 36,
        "archaic": "Bu eşq mənə həm həyat, həm ölüm verdi.",
        "modern": "Sevgi həm yaşamaq, həm də əzab verdi.",
        "gold_label": "neutral"
    },
    {
        "id": 37,
        "archaic": "Aşiq olan bəndə səbr ilə tanınar.",
        "modern": "Sevgisi həqiqi olan insan səbirli olar.",
        "gold_label": "neutral"
    },
    {
        "id": 38,
        "archaic": "Eşq sirrini könüldə saxlamaq gərəkdir.",
        "modern": "Sevgi sirrini ürəkdə saxlamaq lazımdır.",
        "gold_label": "neutral"
    },
    {
        "id": 39,
        "archaic": "Bu könül sənə əsir oldu.",
        "modern": "Ürəyim sənə bağlandı.",
        "gold_label": "positive"
    },
    {
        "id": 40,
        "archaic": "Eşq bəlası şirindir.",
        "modern": "Sevginin verdiyi əzab belə xoşdur.",
        "gold_label": "positive"
    },
    {
        "id": 41,
        "archaic": "Hər dərdin bir səbəbi var, bu dərdin adı eşqdir.",
        "modern": "Bu ağrının səbəbi sevgidir.",
        "gold_label": "neutral"
    },
    {
        "id": 42,
        "archaic": "Bu yolun sonu ya vüsaldır, ya həlak.",
        "modern": "Bu yol ya qovuşma ilə, ya da məhv ilə bitir.",
        "gold_label": "neutral"
    },
    {
        "id": 43,
        "archaic": "Aşiq üçün səbr zamanla ölçülməz.",
        "modern": "Sevən insan üçün zaman anlayışı dəyişir.",
        "gold_label": "neutral"
    },
    {
        "id": 44,
        "archaic": "Bu könül səni gördükdən sonra özgə bilmədi.",
        "modern": "Səni gördükdən sonra başqasını düşünə bilmədim.",
        "gold_label": "positive"
    },
    {
        "id": 45,
        "archaic": "Eşq məni səssiz fəryada saldı.",
        "modern": "Sevgi məni içimdə qışqıran hala gətirdi.",
        "gold_label": "negative"
    },
    {
        "id": 46,
        "archaic": "Bu dərdin dərmanı yenə bu dərddir.",
        "modern": "Bu ağrının yeganə çarəsi elə özüdür.",
        "gold_label": "neutral"
    },
    {
        "id": 47,
        "archaic": "Aşiq olan canını da əsirgəməz.",
        "modern": "Sevən insan həyatını belə əsirgəməz.",
        "gold_label": "positive"
    },
    {
        "id": 48,
        "archaic": "Bu eşq yükü ağırdır, amma atılmaz.",
        "modern": "Sevgi çətindir, amma ondan imtina edilmir.",
        "gold_label": "positive"
    },
    {
        "id": 49,
        "archaic": "Könül eşq ilə dirilər.",
        "modern": "Ürək sevgi ilə canlanır.",
        "gold_label": "positive"
    },
    {
        "id": 50,
        "archaic": "Eşq mənə nə verdin desələr, dərd deyərəm.",
        "modern": "Məndən soruşsalar ki, sevgi sənə nə verdi, cavabım dərd olar.",
        "gold_label": "negative"
    }
]


In [6]:
xlmr_name = "xlm-roberta-base"
mbert_name = "bert-base-multilingual-cased"

xlmr_tok = AutoTokenizer.from_pretrained(xlmr_name)
xlmr_emb = AutoModel.from_pretrained(xlmr_name).to(device)

mbert_tok = AutoTokenizer.from_pretrained(mbert_name)
mbert_emb = AutoModel.from_pretrained(mbert_name).to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

In [7]:
def mean_pooling(output, mask):
    token_embeddings = output.last_hidden_state
    mask = mask.unsqueeze(-1).expand(token_embeddings.size())
    return (token_embeddings * mask).sum(1) / torch.clamp(mask.sum(1), min=1e-9)

def encode(texts, tokenizer, model, batch_size=16):
    model.eval()
    all_emb = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            output = model(**encoded)
            pooled = mean_pooling(output, encoded["attention_mask"])
            all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

In [8]:
archaic_texts = [p["archaic"] for p in pairs]
modern_texts  = [p["modern"] for p in pairs]

xlmr_arch = xlmr_sent(archaic_texts, batch_size=16)
xlmr_mod  = xlmr_sent(modern_texts, batch_size=16)

mbert_arch = mbert_sent(archaic_texts, batch_size=16)
mbert_mod  = mbert_sent(modern_texts, batch_size=16)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [9]:
xlmr_arch_emb = encode(archaic_texts, xlmr_tok, xlmr_emb)
xlmr_mod_emb  = encode(modern_texts,  xlmr_tok, xlmr_emb)

mbert_arch_emb = encode(archaic_texts, mbert_tok, mbert_emb)
mbert_mod_emb  = encode(modern_texts,  mbert_tok, mbert_emb)

In [10]:
rows = []
for i, p in enumerate(pairs):
    rows.append({
        "text_archaic": p["archaic"],
        "text_modern": p["modern"],
        "gold_label": p["gold_label"],

        "xlmr_arch_label": xlmr_arch[i]["label"],
        "xlmr_mod_label":  xlmr_mod[i]["label"],
        "mbert_arch_label": mbert_arch[i]["label"],
        "mbert_mod_label":  mbert_mod[i]["label"],

        "xlmr_similarity": cosine_sim(xlmr_arch_emb[i], xlmr_mod_emb[i]),
        "mbert_similarity": cosine_sim(mbert_arch_emb[i], mbert_mod_emb[i]),
    })

df = pd.DataFrame(rows)

In [11]:
df

,text_archaic,text_modern,gold_label,xlmr_arch_label,xlmr_mod_label,mbert_arch_label,mbert_mod_label,xlmr_similarity,mbert_similarity
0,Məndə Mecnundan füzun aşiqlik istidadı var.,Məndə Məcnundan da artıq aşiq olmaq qabiliyyət...,positive,negative,negative,negative,neutral,0.994589,0.851754
1,"Məni candan usandırdı, cəfaindən usanmazmı?","Mən əzabdan bezdim, bəs o zülm etməkdən bezməzmi?",negative,negative,negative,negative,negative,0.997232,0.817989
2,Dərdimi deməyə bir həmdəm tapılmaz.,Dərdimi bölüşəcəyim bir yaxın insan yoxdur.,negative,negative,negative,negative,negative,0.997585,0.839076
3,"Eşqdir ol kim, varlıqdan alar nişanı.",Eşq insanın bütün varlığını dəyişdirən bir hal...,neutral,negative,neutral,neutral,neutral,0.996662,0.717457
4,Aşiqlərə hər dəm bəladır səfa.,Aşiqlər üçün rahatlıq belə əzab verir.,negative,neutral,positive,negative,negative,0.998044,0.804631
5,"Könül kim, eşqə düşdü, səbrü qərar itirdi.",Ürək eşqə düşəndə səbrini və rahatlığını itirir.,negative,negative,positive,negative,negative,0.997611,0.797771
6,Bu hicran atəşi yandırdı məni.,Bu ayrılıq məni daxildən yandırdı.,negative,negative,negative,negative,negative,0.997762,0.765450
7,Sənsiz bu könül viranədir.,Sənsiz ürəyim boş və dağılmış kimidir.,negative,negative,negative,negative,negative,0.996618,0.740123
8,Eşq yolunda can vermək asandır.,Sevgi uğrunda həyatını qurban vermək asandır.,positive,positive,positive,negative,negative,0.999068,0.793623
9,Göz yaşı könlün sözüdür.,Göz yaşları insanın hisslərini ifadə edir.,neutral,neutral,neutral,positive,neutral,0.994537,0.671060


In [12]:
df["xlmr_arch_correct"]  = df["xlmr_arch_label"]  == df["gold_label"]
df["xlmr_mod_correct"]   = df["xlmr_mod_label"]   == df["gold_label"]
df["mbert_arch_correct"] = df["mbert_arch_label"] == df["gold_label"]
df["mbert_mod_correct"]  = df["mbert_mod_label"]  == df["gold_label"]

df["xlmr_consistent"]  = df["xlmr_arch_label"]  == df["xlmr_mod_label"]
df["mbert_consistent"] = df["mbert_arch_label"] == df["mbert_mod_label"]

print("\n=== ACCURACY ===")
print("XLM-R archaic:", df["xlmr_arch_correct"].mean())
print("XLM-R modern :", df["xlmr_mod_correct"].mean())
print("mBERT archaic:", df["mbert_arch_correct"].mean())
print("mBERT modern :", df["mbert_mod_correct"].mean())

print("\n=== CONSISTENCY ===")
print("XLM-R:", df["xlmr_consistent"].mean())
print("mBERT:", df["mbert_consistent"].mean())


=== ACCURACY ===
XLM-R archaic: 0.6
XLM-R modern : 0.6
mBERT archaic: 0.48
mBERT modern : 0.64

=== CONSISTENCY ===
XLM-R: 0.66
mBERT: 0.58


In [13]:
# save results to excel

from openpyxl import load_workbook
from openpyxl.styles import PatternFill

excel_path = "fuzuli_sentiment_similarity_colored.xlsx"
df.to_excel(excel_path, index=False)

wb = load_workbook(excel_path)
ws = wb.active

GREEN = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
RED = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
YELLOW = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")

DARK_GREEN = PatternFill(start_color="63BE7B", end_color="63BE7B", fill_type="solid")
LIGHT_GREEN = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
ORANGE = PatternFill(start_color="F4B084", end_color="F4B084", fill_type="solid")

header = {cell.value: idx + 1 for idx, cell in enumerate(ws[1])}

gold_col = header["gold_label"]

xlmr_arch_col = header["xlmr_arch_label"]
xlmr_mod_col = header["xlmr_mod_label"]

mbert_arch_col = header["mbert_arch_label"]
mbert_mod_col = header["mbert_mod_label"]

xlmr_sim_col = header["xlmr_similarity"]
mbert_sim_col = header["mbert_similarity"]

for row in range(2, ws.max_row + 1):

    gold = ws.cell(row, gold_col).value

    for col in [xlmr_arch_col, xlmr_mod_col, mbert_arch_col, mbert_mod_col]:
        cell = ws.cell(row, col)
        if cell.value == gold:
            cell.fill = GREEN
        else:
            cell.fill = RED

    if ws.cell(row, xlmr_arch_col).value == ws.cell(row, xlmr_mod_col).value:
        ws.cell(row, xlmr_mod_col).fill = GREEN
    else:
        ws.cell(row, xlmr_mod_col).fill = YELLOW

    if ws.cell(row, mbert_arch_col).value == ws.cell(row, mbert_mod_col).value:
        ws.cell(row, mbert_mod_col).fill = GREEN
    else:
        ws.cell(row, mbert_mod_col).fill = YELLOW

    for col in [xlmr_sim_col, mbert_sim_col]:
        sim_cell = ws.cell(row, col)
        sim = float(sim_cell.value)

        if sim >= 0.75:
            sim_cell.fill = DARK_GREEN
        elif sim >= 0.60:
            sim_cell.fill = LIGHT_GREEN
        elif sim >= 0.45:
            sim_cell.fill = ORANGE
        else:
            sim_cell.fill = RED

wb.save(excel_path)
print(f"Saved color-coded file: {excel_path}")


Saved color-coded file: fuzuli_sentiment_similarity_colored.xlsx


In [14]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

labels = ["negative", "neutral", "positive"]

#xlmr metrics
print("\n=== XLM-R (ARCHAIC) ===")
print(classification_report(
    df["gold_label"],
    df["xlmr_arch_label"],
    labels=labels,
    digits=4
))

print("\n=== XLM-R (MODERN) ===")
print(classification_report(
    df["gold_label"],
    df["xlmr_mod_label"],
    labels=labels,
    digits=4
))

#mbert metrics
print("\n=== mBERT (ARCHAIC) ===")
print(classification_report(
    df["gold_label"],
    df["mbert_arch_label"],
    labels=labels,
    digits=4
))

print("\n=== mBERT (MODERN) ===")
print(classification_report(
    df["gold_label"],
    df["mbert_mod_label"],
    labels=labels,
    digits=4
))



=== XLM-R (ARCHAIC) ===
              precision    recall  f1-score   support

    negative     0.6562    0.8400    0.7368        25
     neutral     0.4000    0.5455    0.4615        11
    positive     1.0000    0.2143    0.3529        14

    accuracy                         0.6000        50
   macro avg     0.6854    0.5332    0.5171        50
weighted avg     0.6961    0.6000    0.5688        50


=== XLM-R (MODERN) ===
              precision    recall  f1-score   support

    negative     0.7083    0.6800    0.6939        25
     neutral     0.4615    0.5455    0.5000        11
    positive     0.5385    0.5000    0.5185        14

    accuracy                         0.6000        50
   macro avg     0.5694    0.5752    0.5708        50
weighted avg     0.6065    0.6000    0.6021        50


=== mBERT (ARCHAIC) ===
              precision    recall  f1-score   support

    negative     0.5429    0.7600    0.6333        25
     neutral     0.3636    0.3636    0.3636        11
 

In [15]:
def prf(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro"
    )
    return p, r, f

metrics = []

metrics.append(("XLM-R", "archaic", *prf(df["gold_label"], df["xlmr_arch_label"])))
metrics.append(("XLM-R", "modern",  *prf(df["gold_label"], df["xlmr_mod_label"])))
metrics.append(("mBERT", "archaic", *prf(df["gold_label"], df["mbert_arch_label"])))
metrics.append(("mBERT", "modern",  *prf(df["gold_label"], df["mbert_mod_label"])))

metrics_df = pd.DataFrame(
    metrics,
    columns=["model", "text_type", "precision", "recall", "f1"]
)

metrics_df.to_csv("fuzuli_prf_metrics.csv", index=False)
print(metrics_df)


   model text_type  precision    recall        f1
0  XLM-R   archaic   0.685417  0.533247  0.517107
1  XLM-R    modern   0.569444  0.575152  0.570799
2  mBERT   archaic   0.385498  0.398355  0.369360
3  mBERT    modern   0.666667  0.604329  0.587903
